In [ ]:
import tiktoken
import pandas as pd
import os
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
from tqdm import tqdm
import numpy as np
import faiss

import pickle

load_dotenv()
tqdm.pandas()


from create_knowledge_database import create_embed,build_index,merge_all_faiss_index,search_bm_25,search_in_faiss

In [18]:
token=os.getenv('HF_TOKEN')
enc = tiktoken.get_encoding('cl100k_base')
MODELS = {
    'bge-m3': {'model': SentenceTransformer("BAAI/bge-m3",token=token)},
    'multi-e5': {'model': SentenceTransformer("intfloat/multilingual-e5-base",token=token)}
}

## BGE-M3

In [6]:
create_embed('parsed_pages/hunter_parsed_pages.jsonl',
                 'chunks/hunter_chunks.npy',
                 'bm3/embeded_chunks/hunter_embedding_chunks_bge-m3.npy',source='hunter')
arr1=np.load('bm3/embeded_chunks/hunter_embedding_chunks_bge-m3.npy')
arr2=np.load('chunks/hunter_chunks.npy',allow_pickle=True)
print(len(arr1),len(arr2))





100%|██████████| 1982/1982 [00:01<00:00, 1179.92it/s]


Batches:   0%|          | 0/200 [00:00<?, ?it/s]

6391 6391


100%|██████████| 7655/7655 [00:03<00:00, 2402.59it/s]


Batches:   0%|          | 0/475 [00:00<?, ?it/s]

100%|██████████| 1102/1102 [00:00<00:00, 1301.18it/s]


Batches:   0%|          | 0/107 [00:00<?, ?it/s]

In [ ]:
create_embed('parsed_pages/naruto_parsed_pages.jsonl',
                 'chunks/naruto_chunks.npy',
                 'bm3/embeded_chunks/naruto_embedding_chunks_bge-m3.npy',source='naruto')
arr1=np.load('bm3/embeded_chunks/naruto_embedding_chunks_bge-m3.npy')
arr2=np.load('chunks/naruto_chunks.npy',allow_pickle=True)
print(len(arr1),len(arr2))

In [ ]:
create_embed('parsed_pages/sao_parsed_pages.jsonl',
                 'chunks/sao_chunks.npy',
                 'bm3/embeded_chunks/sao_embedding_chunks_bge-m3.npy',source='sao')
arr1=np.load('bm3/embeded_chunks/sao_embedding_chunks_bge-m3.npy')
arr2=np.load('chunks/sao_chunks.npy',allow_pickle=True)
print(len(arr1),len(arr2))

In [ ]:
build_index('bm3/embeded_chunks/hunter_embedding_chunks_bge-m3.npy','bm3/faiss/hunter_bge-m3.faiss')

In [ ]:
build_index('bm3/embeded_chunks/naruto_embedding_chunks_bge-m3.npy','bm3/faiss/naruto_bge-m3.faiss')

In [ ]:
build_index('bm3/embeded_chunks/sao_embedding_chunks_bge-m3.npy','bm3/faiss/sao_bge-m3.faiss')

In [ ]:
merge_all_faiss_index(list_path_faiss_dataset=['bm3/faiss/hunter_bge-m3.faiss','bm3/faiss/naruto_bge-m3.faiss','bm3/faiss/sao_bge-m3.faiss'],is_save=True,path_to_save='bm3/faiss/all_fandom_bge-m3.faiss')

## E5-LARGE

In [ ]:
create_embed('parsed_pages/hunter_parsed_pages.jsonl',
                 'chunks/hunter_chunks.npy',
                 'multi-e5/embed_chunks/hunter_embedding_chunks_e5-large.npy',source='hunter',retriever_model='multi-e5')
arr1=np.load('multi-e5/embed_chunks/hunter_embedding_chunks_e5-large.npy')
arr2=np.load('chunks/hunter_chunks.npy',allow_pickle=True)
print(len(arr1),len(arr2))
print(arr1.shape)
    

In [ ]:
create_embed('parsed_pages/naruto_parsed_pages.jsonl',
                 'chunks/naruto_chunks.npy',
                 'multi-e5/embed_chunks/naruto_embedding_chunks_e5-large.npy',source='naruto',retriever_model='multi-e5')
arr1=np.load('multi-e5/embed_chunks/naruto_embedding_chunks_e5-large.npy')
arr2=np.load('chunks/naruto_chunks.npy',allow_pickle=True)
print(len(arr1),len(arr2))

In [ ]:
create_embed('parsed_pages/sao_parsed_pages.jsonl',
                 'chunks/sao_chunks.npy',
                 'multi-e5/embed_chunks/sao_embedding_chunks_e5-large.npy',source='sao',retriever_model='multi-e5')
arr1=np.load('multi-e5/embed_chunks/sao_embedding_chunks_e5-large.npy')
arr2=np.load('chunks/sao_chunks.npy',allow_pickle=True)
print(len(arr1),len(arr2))

In [ ]:
build_index('multi-e5/embed_chunks/hunter_embedding_chunks_e5-large.npy','multi-e5/faiss/hunter_e5-large.faiss')
build_index('multi-e5/embed_chunks/naruto_embedding_chunks_e5-large.npy','multi-e5/faiss/naruto_e5-large.faiss')
build_index('multi-e5/embed_chunks/sao_embedding_chunks_e5-large.npy','multi-e5/faiss/sao_e5-large.faiss')

In [ ]:
merge_all_faiss_index(list_path_faiss_dataset=['multi-e5/faiss/hunter_e5-large.faiss',
                                               'multi-e5/faiss/naruto_e5-large.faiss',
                                               'multi-e5/faiss/sao_e5-large.faiss'],is_save=True,path_to_save='multi-e5/faiss/all_fandom_e5-large.faiss')

## CREATE EVAL SET

In [ ]:
data=pd.read_json('eval_set_v3_clean.jsonl',lines=True)

In [ ]:
with open('chunks/all_fandom_chunks.pkl','rb') as f:
    all_chunks=pickle.load(f)
tokenized_chunks=[c['text'].lower().split() for c in all_chunks]
faiss_index_bge_m3=faiss.read_index('bm3/faiss/all_fandom_bge-m3.faiss')
faiss_index_e5=faiss.read_index('multi-e5/faiss/all_fandom_e5-large.faiss')

In [14]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 338 entries, 0 to 337
Data columns (total 17 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   question_id          338 non-null    object 
 1   chunk_id             317 non-null    float64
 2   source_chunk_title   338 non-null    object 
 3   question             338 non-null    object 
 4   ground_truth_answer  338 non-null    object 
 5   question_type        338 non-null    object 
 6   notes                66 non-null     object 
 7   relevant_chunk_ids   338 non-null    object 
 8   language             338 non-null    object 
 9   ground_truth_text    338 non-null    object 
 10  answer_type          338 non-null    object 
 11  reasoning_chain      19 non-null     object 
 12  negative_type        25 non-null     object 
 13  base_question_id     20 non-null     object 
 14  referenced_title     6 non-null      object 
 15  false_assumption     4 non-null      obj

In [24]:
results_bge_m3=data.progress_apply(lambda x: search_in_faiss(
                                                                x['question'],
                                                                top_k=20,
                                                                faiss_index=faiss_index_bge_m3,
                                                                all_chunks=all_chunks,
                                                                threshold=None,
                                                                return_time=True,
                                                                retriever_model='bge-m3'


),axis=1)
data['bge_m3_chunks'],data['bge_m3_chunks_time']=zip(*results_bge_m3)

100%|██████████| 338/338 [00:09<00:00, 35.93it/s]


In [25]:
results_e5=data.progress_apply(lambda x: search_in_faiss(
                                                                x['question'],
                                                                top_k=20,
                                                                faiss_index=faiss_index_e5,
                                                                all_chunks=all_chunks,
                                                                threshold=None,
                                                                return_time=True,
                                                                retriever_model='multi-e5'


),axis=1)
data['e5_chunks'],data['e5_chunks_time']=zip(*results_e5)

100%|██████████| 338/338 [00:06<00:00, 55.83it/s]


In [26]:
results_bm_25=data.progress_apply(lambda x: search_bm_25(
                                                                x['question'],
                                                                top_k=20,
                                                                tokenized_chunks=tokenized_chunks,
                                                                all_chunks=all_chunks,
                                                                
                                                                return_time=True,
                                                               


),axis=1)
data['bm_25_chunks'],data['bm_25_chunks_time']=zip(*results_bm_25)

100%|██████████| 338/338 [10:00<00:00,  1.78s/it]


In [28]:
data['union_all_top_chunks'] = data.apply(lambda x: list({
    c['text']: c for c in x['bge_m3_chunks'] + x['e5_chunks'] + x['bm_25_chunks']
}.values()), axis=1)

In [29]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 338 entries, 0 to 337
Data columns (total 24 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   question_id           338 non-null    object 
 1   chunk_id              317 non-null    float64
 2   source_chunk_title    338 non-null    object 
 3   question              338 non-null    object 
 4   ground_truth_answer   338 non-null    object 
 5   question_type         338 non-null    object 
 6   notes                 66 non-null     object 
 7   relevant_chunk_ids    338 non-null    object 
 8   language              338 non-null    object 
 9   ground_truth_text     338 non-null    object 
 10  answer_type           338 non-null    object 
 11  reasoning_chain       19 non-null     object 
 12  negative_type         25 non-null     object 
 13  base_question_id      20 non-null     object 
 14  referenced_title      6 non-null      object 
 15  false_assumption      4

In [35]:
data.to_csv('extended_eval_set_with_combined_chunks1.csv')